# 04 — ML Model: Feature Engineering & Target Variable (GBP/IDR)

Builds the feature set (lagged returns, rolling volatility, RSI, MACD, plus the
Week 2 percentile rank / z-score) and the target variable (does the rate rise
over the next N days). Model training comes after this — this notebook stops
right after the target is built and sanity-checked, since that's the part most
worth slowing down for.


In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data import get_data_incremental
from src.signals import add_percentile_rank, add_zscore, classify_signal
from src.features import lagged_returns, rolling_volatility, rsi, macd

plt.rcParams["figure.figsize"] = (12, 5)


## 1. Load data + rebuild the Week 2 signal columns

In [2]:
PERIOD = 180

df = get_data_incremental(start="2015-01-01")
df = add_percentile_rank(df, period=PERIOD)
df = add_zscore(df, period=PERIOD)
df["Value_Signal"] = df["Percentile_Rank"].apply(classify_signal)

print(df.shape)
df.tail()


No cache found. Fetching full history...
[Falling back to SYNTHETIC data] Reason: No module named 'yfinance'
Cache created (synthetic data), 2610 rows.


(2610, 7)


,Open,High,Low,Close,Percentile_Rank,Zscore,Value_Signal
2024-12-26,11982.791872,12017.950416,11985.581098,12001.843951,0.888889,1.263271,Unfavorable
2024-12-27,12015.024860,12045.986234,11982.387942,12002.277157,0.888889,1.249216,Unfavorable
2024-12-30,11908.346556,11924.232722,11889.590246,11908.248730,0.827778,1.095059,Unfavorable
2024-12-31,11839.967626,11867.048264,11812.572248,11847.943206,0.822222,0.993353,Unfavorable
2025-01-01,11979.821076,11992.679713,11971.700950,11977.540412,0.866667,1.174775,Unfavorable


## 2. Add features

Lagged returns at three horizons, rolling volatility at two windows, RSI, and MACD.
`Percentile_Rank` and `Zscore` from Week 2 are also used as model features here,
not just as display labels.

In [3]:
df["ret_1d"] = lagged_returns(df, 1)
df["ret_5d"] = lagged_returns(df, 5)
df["ret_20d"] = lagged_returns(df, 20)

df["vol_5d"] = rolling_volatility(df, 5)
df["vol_20d"] = rolling_volatility(df, 20)

df["rsi_14"] = rsi(df, period=14)

macd_df = macd(df)
df["macd"] = macd_df["MACD"]
df["macd_signal"] = macd_df["Signal"]

feature_cols = [
    "ret_1d", "ret_5d", "ret_20d",
    "vol_5d", "vol_20d",
    "rsi_14", "macd", "macd_signal",
    "Percentile_Rank", "Zscore",
]

df[feature_cols].tail()


,ret_1d,ret_5d,ret_20d,vol_5d,vol_20d,rsi_14,macd,macd_signal,Percentile_Rank,Zscore
2024-12-26,0.002492,0.003408,-0.015415,0.006959,0.007887,33.868104,-16.135760,38.141646,0.888889,1.263271
2024-12-27,0.000036,0.013348,-0.027830,0.004038,0.007221,25.405449,-19.021915,26.708934,0.888889,1.249216
2024-12-30,-0.007834,-0.002925,-0.041440,0.004757,0.007123,23.422956,-28.567217,15.653703,0.827778,1.095059
2024-12-31,-0.005064,-0.012420,-0.052319,0.004079,0.006863,22.808290,-40.530875,4.416788,0.822222,0.993353
2025-01-01,0.010938,0.000462,-0.036637,0.007294,0.007463,35.437852,-39.103983,-4.287366,0.866667,1.174775


**Sanity check:** every feature should show `NaN` only during its own warm-up
period, and be a real number after that. Quick scan before moving on.

In [4]:
df[feature_cols].isna().sum()


ret_1d               1
ret_5d               5
ret_20d             20
vol_5d               5
vol_20d             20
rsi_14              13
macd                 0
macd_signal          0
Percentile_Rank    179
Zscore             179
dtype: int64

## 3. Create the target variable

**This is the step most worth being careful with.** The target legitimately looks
*forward* in time (`Close.shift(-HORIZON)`) — that's what makes it a label at all,
not a bug. The lookahead rule only applies to *features*: none of the columns
above use a negative shift, and none of them should ever be allowed to.

Target definition: does `Close` HORIZON trading days from now end up higher than
`Close` today?

In [5]:
HORIZON = 10  # trading days ahead — adjust and re-run to compare later if you want

df["future_close"] = df["Close"].shift(-HORIZON)
df["target"] = (df["future_close"] > df["Close"]).astype(int)

# the last HORIZON rows have no real future value to compare against — the shift
# pulled in NaN for those, so future_close/target for them are meaningless
print("Rows with no valid target (expect exactly HORIZON):", df["future_close"].isna().sum())
df[["Close", "future_close", "target"]].tail(HORIZON + 3)


Rows with no valid target (expect exactly HORIZON): 10


,Close,future_close,target
2024-12-16,12184.285469,11908.248730,0
2024-12-17,12095.130489,11847.943206,0
2024-12-18,11935.088849,11977.540412,1
2024-12-19,11961.081347,NaN,0
2024-12-20,11844.185044,NaN,0
2024-12-23,11943.182471,NaN,0
2024-12-24,11996.947610,NaN,0
2024-12-25,11972.013174,NaN,0
2024-12-26,12001.843951,NaN,0
2024-12-27,12002.277157,NaN,0


## 4. Manual spot-check — don't just trust the code ran

Pick a handful of rows well before the end of the dataset and verify by eye that
`target` matches what a human would conclude by comparing `Close` to `future_close`.

In [6]:
check_rows = df.iloc[500:505][["Close", "future_close", "target"]]
print(check_rows)
print()
for date, row in check_rows.iterrows():
    expected = int(row["future_close"] > row["Close"])
    print(f"{date.date()}: Close={row['Close']:.2f}  future_close={row['future_close']:.2f}  "
          f"target={int(row['target'])}  manual_expected={expected}  match={row['target']==expected}")


                   Close  future_close  target
2016-12-01  19019.005961  19216.436802       1
2016-12-02  19027.764132  18951.209854       0
2016-12-05  19121.304709  18964.681207       0
2016-12-06  19203.760143  18991.401524       0
2016-12-07  19271.610653  19061.215844       0

2016-12-01: Close=19019.01  future_close=19216.44  target=1  manual_expected=1  match=True
2016-12-02: Close=19027.76  future_close=18951.21  target=0  manual_expected=0  match=True
2016-12-05: Close=19121.30  future_close=18964.68  target=0  manual_expected=0  match=True
2016-12-06: Close=19203.76  future_close=18991.40  target=0  manual_expected=0  match=True
2016-12-07: Close=19271.61  future_close=19061.22  target=0  manual_expected=0  match=True


## 5. Check the target's class balance

Important context before evaluating any model later: if this is heavily skewed
toward one class, the naive baseline accuracy will already be high, and a model
needs to clear *that*, not 50%.

In [7]:
print(df["target"].value_counts(dropna=False))
print()
print("Proportion positive (rate goes up):", df["target"].mean())


target
0    1340
1    1270
Name: count, dtype: int64

Proportion positive (rate goes up): 0.48659003831417624


## 6. Drop rows that can't be used

- The warm-up period at the start (features not fully formed yet)
- The last `HORIZON` rows at the end (no valid target)

In [8]:
model_df = df.dropna(subset=feature_cols + ["target"]).copy()
print(f"Original rows: {len(df)}  ->  usable rows: {len(model_df)}")
print("Date range:", model_df.index.min().date(), "to", model_df.index.max().date())


Original rows: 2610  ->  usable rows: 2431
Date range: 2015-09-09 to 2025-01-01


## Next steps

- Chronological train/test split on `model_df` (no shuffling)
- Naive majority-class baseline on the test set
- Train logistic regression and XGBoost classifiers
- Evaluate against the baseline, check feature importances
- Write an honest conclusion — given Week 3's Option C result was close to
  a zero real-world edge, a modest improvement here would be a genuinely
  good outcome, not a disappointing one.
